# Pandas II - Continued

**DA2402 · Dr. Arun B Ayyar**

 This notebook covers following

| § | Topic |
|---|---|
| 1 | Building `Series` and `DataFrame` from scratch — and **index alignment** |
| 2 | First look: `.info()`, `.describe()`, `.sample()`, `.nunique()` |
| 3 | Descriptive statistics, `corr`, sorting, ranking, `nlargest` |
| 4 | **Missing data** — `isna`, `dropna`, `fillna`, and how `NaN` behaves in aggregations |
| 5 | Duplicates — `duplicated`, `drop_duplicates` |
| 6 | String methods — the `.str` accessor |
| 7 | Binning and recoding — `cut`, `qcut`, `map`, `where`, `apply` |
| 8 | **Dates and times** — `to_datetime`, `.dt`, `resample`, `shift`, `pct_change`, `rolling` |
| 9 | **Reshaping** — `pivot`, `pivot_table`, `melt`, `stack`/`unstack`, `crosstab` |
| 10 | `groupby` in depth — multiple keys, `transform`, `filter`, named aggregation |
| 11 | File I/O — `read_csv` options and `to_csv` |
| 12 | Copy vs view, `SettingWithCopyWarning`, and `.assign()` chaining |
| 13 | `query()`, categoricals, and memory |

Same two data files as `pandas1`. Everything else is generated inside the notebook,
deterministically — run it twice and you get the same numbers.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 30)

print("pandas", pd.__version__, "| numpy", np.__version__)

pandas 2.2.2 | numpy 2.0.2


In [6]:
nifty50 = pd.read_csv('https://raw.githubusercontent.com/iitm-da/da2402/master/intro/nifty50.csv')

# tidy the column names (same recipe as pandas1)
new_names = {c: c.replace("\n", "").strip().title().split("(")[0].strip() for c in nifty50.columns}
nifty50 = nifty50.rename(columns=new_names).rename(columns={"365 D % Chng  02-Aug-2024": "365 D %Chng"})

# every numeric-looking column arrives as text with thousands separators
num_cols = ["Open", "High", "Low", "Prev. Close", "Ltp", "Indicative Close",
            "Chng", "%Chng", "Volume", "Value", "52W H", "52W L",
            "30 D   %Chng", "365 D %Chng"]
for c in num_cols:
    nifty50[c] = pd.to_numeric(nifty50[c].astype(str).str.replace(",", "", regex=False), errors="coerce")

nifty50["Symbol"] = nifty50["Symbol"].str.strip()

index_row = nifty50[nifty50["Symbol"] == "NIFTY 50"]        # the benchmark itself
nifty = nifty50[nifty50["Symbol"] != "NIFTY 50"].copy()     # the 50 constituents

print("rows in file:", len(nifty50), "| constituents:", len(nifty))
nifty.head(3)

rows in file: 51 | constituents: 50


,Symbol,Open,High,Low,Prev. Close,Ltp,Indicative Close,Chng,%Chng,Volume,Value,52W H,52W L,30 D %Chng,365 D %Chng
1,TITAN,3356.3,3443.0,3348.00,3356.30,3424.0,NaN,67.70,2.02,837335,285.13,3867.0,2925.00,-7.29,-1.28
2,INDUSINDBK,820.0,848.7,810.05,804.05,817.4,NaN,13.35,1.66,18982770,1570.06,1498.0,606.00,-4.33,-41.67
3,SBILIFE,1833.3,1860.4,1818.70,1831.50,1860.0,NaN,28.50,1.56,1251783,231.37,1936.0,1372.55,3.18,6.42


In [7]:
# Bring in the sector master and keep one tidy frame for the whole notebook.
sectors = pd.read_csv('https://raw.githubusercontent.com/iitm-da/da2402/master/intro/ind_nifty50list.csv')
sectors["Symbol"] = sectors["Symbol"].str.strip()

df = nifty.merge(sectors[["Symbol", "Company Name", "Industry"]], on="Symbol", how="left")
print(df.shape)
df.head(3)

(50, 17)


,Symbol,Open,High,Low,Prev. Close,Ltp,Indicative Close,Chng,%Chng,Volume,Value,52W H,52W L,30 D %Chng,365 D %Chng,Company Name,Industry
0,TITAN,3356.3,3443.0,3348.00,3356.30,3424.0,NaN,67.70,2.02,837335,285.13,3867.0,2925.00,-7.29,-1.28,Titan Company Ltd.,Consumer Durables
1,INDUSINDBK,820.0,848.7,810.05,804.05,817.4,NaN,13.35,1.66,18982770,1570.06,1498.0,606.00,-4.33,-41.67,IndusInd Bank Ltd.,Financial Services
2,SBILIFE,1833.3,1860.4,1818.70,1831.50,1860.0,NaN,28.50,1.56,1251783,231.37,1936.0,1372.55,3.18,6.42,SBI Life Insurance Company Ltd.,Financial Services


## 1 · Building a `Series` and a `DataFrame` from scratch



In [3]:
# A Series from a list: pandas supplies a default RangeIndex.
s1 = pd.Series([120.5, 98.0, 143.25, 87.4])
print(s1, "\n")

# A Series with an explicit index and a name.
s2 = pd.Series([120.5, 98.0, 143.25, 87.4],
               index=["INFY", "TCS", "WIPRO", "HCLTECH"],
               name="price")
print(s2)
print("\n.values ->", s2.values)
print(".index  ->", list(s2.index))
print(".name   ->", s2.name)
print("dtype   ->", s2.dtype)

0    120.50
1     98.00
2    143.25
3     87.40
dtype: float64 

INFY       120.50
TCS         98.00
WIPRO      143.25
HCLTECH     87.40
Name: price, dtype: float64

.values -> [120.5   98.   143.25  87.4 ]
.index  -> ['INFY', 'TCS', 'WIPRO', 'HCLTECH']
.name   -> price
dtype   -> float64


In [ ]:
# A Series from a dict: the keys BECOME the index.
s3 = pd.Series({"INFY": 120.5, "TCS": 98.0, "WIPRO": 143.25})
print(s3)

# Indexing works by label, and also positionally via .iloc
print("\nby label    s3['TCS'] =", s3["TCS"])
print("positional  s3.iloc[1] =", s3.iloc[1])

INFY     120.50
TCS       98.00
WIPRO    143.25
dtype: float64

by label    s3['TCS'] = 98.0
positional  s3.iloc[1] = 98.0


In [4]:
d={"INFY": 120.5, "TCS": 98.0, "WIPRO": 143.25}
s31 = pd.Series(data=d, index=["x", "y", "z"])
print(s31)
# Index is first built with the keys from the dictionary.
#After this the Series is reindexed with the given Index values,
#hence we get all NaN as a result.

x   NaN
y   NaN
z   NaN
dtype: float64


###  **Index alignment**

Arithmetic between two Series lines them up **by index label**, not by position.
Labels present in one but not the other produce `NaN`. This is what makes pandas
different from a NumPy array

In [ ]:
a = pd.Series({"INFY": 100, "TCS": 200, "WIPRO": 300})
b = pd.Series({"WIPRO": 1, "TCS": 2, "HCLTECH": 3})   # different order, different membership

print("a + b  (aligned by label, not position):")
print(a + b)

print("\nfill the gaps instead of propagating NaN:")
print(a.add(b, fill_value=0))

print("\ncompare with NumPy, which aligns by POSITION:")
print(a.values + np.array([1, 2, 3]))

a + b  (aligned by label, not position):
HCLTECH      NaN
INFY         NaN
TCS        202.0
WIPRO      301.0
dtype: float64

fill the gaps instead of propagating NaN:
HCLTECH      3.0
INFY       100.0
TCS        202.0
WIPRO      301.0
dtype: float64

compare with NumPy, which aligns by POSITION:
[101 202 303]


In [ ]:
# Three ways to build a DataFrame.
from_dict = pd.DataFrame({
    "symbol": ["INFY", "TCS", "WIPRO"],
    "price":  [120.5, 98.0, 143.25],
    "sector": ["IT", "IT", "IT"],
})

from_records = pd.DataFrame([
    {"symbol": "INFY",  "price": 120.5},
    {"symbol": "TCS",   "price": 98.0},
    {"symbol": "WIPRO", "price": 143.25},
])

from_array = pd.DataFrame(np.arange(12).reshape(4, 3),
                          columns=["a", "b", "c"],
                          index=["w", "x", "y", "z"])

print(from_dict, "\n")
print(from_records, "\n")
print(from_array)

  symbol   price sector
0   INFY  120.50     IT
1    TCS   98.00     IT
2  WIPRO  143.25     IT 

  symbol   price
0   INFY  120.50
1    TCS   98.00
2  WIPRO  143.25 

   a   b   c
w  0   1   2
x  3   4   5
y  6   7   8
z  9  10  11


In [ ]:
# A DataFrame really is a dict of Series sharing one index.
print("one column is a Series :", type(from_dict["price"]))
print("two columns is a frame :", type(from_dict[["symbol", "price"]]))
print("\nthe frame's index is shared by every column:")
print(from_array.index, "\n")
print("column 'b' as a Series:\n", from_array["b"])

one column is a Series : <class 'pandas.core.series.Series'>
two columns is a frame : <class 'pandas.core.frame.DataFrame'>

the frame's index is shared by every column:
Index(['w', 'x', 'y', 'z'], dtype='object') 

column 'b' as a Series:
 w     1
x     4
y     7
z    10
Name: b, dtype: int64


## 2 · The first look at any new dataset

Before you compute anything, run these. They catch most
data problems immediately

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Symbol            50 non-null     object 
 1   Open              50 non-null     float64
 2   High              50 non-null     float64
 3   Low               50 non-null     float64
 4   Prev. Close       50 non-null     float64
 5   Ltp               50 non-null     float64
 6   Indicative Close  0 non-null      float64
 7   Chng              50 non-null     float64
 8   %Chng             50 non-null     float64
 9   Volume            50 non-null     int64  
 10  Value             50 non-null     float64
 11  52W H             50 non-null     float64
 12  52W L             50 non-null     float64
 13  30 D   %Chng      50 non-null     float64
 14  365 D %Chng       49 non-null     float64
 15  Company Name      50 non-null     object 
 16  Industry          50 non-null     object 
dtyp

`.info()` answers three questions at once: **how many rows**, **which columns have missing
values** (`Non-Null Count` below the row count), and **what dtype each column is**
(`object` almost always means "text, or something that failed to parse").

Notice `Indicative Close`: **0 non-null**. Every value in that column was `-` in the source
file and our `errors="coerce"` turned all of them into `NaN`.

In [ ]:
df.describe()

,Open,High,Low,Prev. Close,Ltp,Indicative Close,Chng,%Chng,Volume,Value,52W H,52W L,30 D %Chng,365 D %Chng
count,50.000000,50.0000,50.000000,50.00000,50.000000,0.0,50.000000,50.000000,5.000000e+01,50.000000,50.000000,50.00000,50.000000,49.000000
mean,2311.517200,2330.0198,2292.603200,2308.40780,2315.236000,NaN,6.828200,-0.014600,5.036996e+06,476.303800,2769.294400,1919.85440,-2.951800,-10.858571
std,2710.907517,2736.4567,2698.156612,2703.50079,2726.421268,NaN,34.348509,0.901342,5.301429e+06,486.020115,3190.315078,2282.68085,5.211051,26.544324
min,159.560000,160.1400,158.280000,159.56000,159.450000,NaN,-48.000000,-1.940000,1.940830e+05,91.650000,170.180000,122.62000,-14.090000,-86.740000
25%,702.950000,704.5625,694.525000,700.72500,698.800000,NaN,-10.375000,-0.647500,8.851970e+05,214.017500,834.425000,555.91250,-5.947500,-20.510000
50%,1474.400000,1482.0500,1458.100000,1474.70000,1470.450000,NaN,-0.320000,-0.120000,2.517863e+06,328.545000,1754.875000,1256.07500,-2.590000,-5.750000
75%,2518.600000,2530.6750,2490.250000,2520.22500,2508.025000,NaN,7.750000,0.535000,8.217320e+06,509.390000,3190.737500,2133.18750,-0.375000,4.530000
max,12400.000000,12577.0000,12370.000000,12365.00000,12543.000000,NaN,178.000000,2.020000,1.898277e+07,2368.550000,13541.650000,10725.00000,15.590000,29.320000


In [ ]:
# .describe() ignores text columns by default. Ask for everything:
df.describe(include="all").T          # .T transposes — usually far easier to read

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Symbol,50,50,TITAN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Open,50.0,NaN,NaN,NaN,2311.5172,2710.907517,159.56,702.95,1474.4,2518.6,12400.0
High,50.0,NaN,NaN,NaN,2330.0198,2736.4567,160.14,704.5625,1482.05,2530.675,12577.0
Low,50.0,NaN,NaN,NaN,2292.6032,2698.156612,158.28,694.525,1458.1,2490.25,12370.0
Prev. Close,50.0,NaN,NaN,NaN,2308.4078,2703.50079,159.56,700.725,1474.7,2520.225,12365.0
Ltp,50.0,NaN,NaN,NaN,2315.236,2726.421268,159.45,698.8,1470.45,2508.025,12543.0
Indicative Close,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Chng,50.0,NaN,NaN,NaN,6.8282,34.348509,-48.0,-10.375,-0.32,7.75,178.0
%Chng,50.0,NaN,NaN,NaN,-0.0146,0.901342,-1.94,-0.6475,-0.12,0.535,2.02
Volume,50.0,NaN,NaN,NaN,5036995.56,5301429.073521,194083.0,885197.0,2517863.0,8217320.0,18982770.0


In [ ]:
#checking the start , end and random samples from the data
print("shape      :", df.shape)
print("dtypes     :\n", df.dtypes.value_counts(), "\n")
print("head(3):");  print(df[["Symbol", "Ltp", "%Chng", "Industry"]].head(3))
print("\ntail(3):"); print(df[["Symbol", "Ltp", "%Chng", "Industry"]].tail(3))
print("\nsample(3, random_state=0) — a RANDOM peek, better than head for spotting oddities:")
print(df[["Symbol", "Ltp", "%Chng", "Industry"]].sample(3, random_state=0))

shape      : (50, 17)
dtypes     :
 float64    13
object      3
int64       1
Name: count, dtype: int64 

head(3):
       Symbol     Ltp  %Chng            Industry
0       TITAN  3424.0   2.02   Consumer Durables
1  INDUSINDBK   817.4   1.66  Financial Services
2     SBILIFE  1860.0   1.56  Financial Services

tail(3):
        Symbol     Ltp  %Chng                    Industry
47        INFY  1460.0  -1.38      Information Technology
48    RELIANCE  1390.0  -1.52  Oil Gas & Consumable Fuels
49  ADANIPORTS  1361.9  -1.94                    Services

sample(3, random_state=0) — a RANDOM peek, better than head for spotting oddities:
        Symbol       Ltp  %Chng                Industry
28    HINDALCO    685.85  -0.27         Metals & Mining
11  ULTRACEMCO  12327.00   0.57  Construction Materials
10        SBIN    800.50   0.60      Financial Services


In [ ]:
# Cardinality: how many distinct values does each column take? #nunique, unique, value_counts
print(df[["Symbol", "Company Name", "Industry"]].nunique(), "\n")
print("the 15 industries:")
print(df["Industry"].unique())
print("\ncounts per industry:")
print(df["Industry"].value_counts())
print("\nas proportions:")
print(df["Industry"].value_counts(normalize=True).round(3).head())

Symbol          50
Company Name    50
Industry        15
dtype: int64 

the 15 industries:
['Consumer Durables' 'Financial Services' 'Automobile and Auto Components'
 'Oil Gas & Consumable Fuels' 'Consumer Services' 'Telecommunication'
 'Construction Materials' 'Information Technology' 'Construction' 'Power'
 'Metals & Mining' 'Fast Moving Consumer Goods' 'Healthcare'
 'Capital Goods' 'Services']

counts per industry:
Industry
Financial Services                12
Automobile and Auto Components     6
Information Technology             5
Fast Moving Consumer Goods         4
Healthcare                         4
Metals & Mining                    4
Oil Gas & Consumable Fuels         3
Power                              2
Construction Materials             2
Consumer Services                  2
Consumer Durables                  2
Telecommunication                  1
Construction                       1
Capital Goods                      1
Services                           1
Name: count, d

## 3 · Descriptive statistics, sorting and ranking

In [ ]:
# Column-wise statistics (axis=0 is the default: collapse DOWN the rows).
stats = df[["Open", "High", "Low", "Ltp", "%Chng", "Volume"]]
print("mean:\n",   stats.mean().round(2), "\n")
print("median:\n", stats.median().round(2), "\n")
print("std:\n",    stats.std().round(2), "\n")
print("quantiles (25/50/75%):")
print(stats.quantile([.25, .5, .75]).round(2))

mean:
 Open         2311.52
High         2330.02
Low          2292.60
Ltp          2315.24
%Chng          -0.01
Volume    5036995.56
dtype: float64 

median:
 Open         1474.40
High         1482.05
Low          1458.10
Ltp          1470.45
%Chng          -0.12
Volume    2517863.00
dtype: float64 

std:
 Open         2710.91
High         2736.46
Low          2698.16
Ltp          2726.42
%Chng           0.90
Volume    5301429.07
dtype: float64 

quantiles (25/50/75%):


         Open     High      Low      Ltp  %Chng     Volume
0.25   702.95   704.56   694.52   698.80  -0.65   885197.0
0.50  1474.40  1482.05  1458.10  1470.45  -0.12  2517863.0
0.75  2518.60  2530.68  2490.25  2508.02   0.54  8217320.0


In [ ]:
# idxmax / idxmin return the LABEL of the extreme value, not the value itself.
best = df["%Chng"].idxmax()
worst = df["%Chng"].idxmin()
print("biggest gainer :", df.loc[best, "Symbol"], df.loc[best, "%Chng"], "%")
print("biggest loser  :", df.loc[worst, "Symbol"], df.loc[worst, "%Chng"], "%")

# nlargest / nsmallest do the same job for the top n, and are much clearer than sort+head.
print("\ntop 5 by % change:")
print(df.nlargest(5, "%Chng")[["Symbol", "Industry", "Ltp", "%Chng"]])
print("\nbottom 5 by % change:")
print(df.nsmallest(5, "%Chng")[["Symbol", "Industry", "Ltp", "%Chng"]])

biggest gainer : TITAN 2.02 %
biggest loser  : ADANIPORTS -1.94 %

top 5 by % change:
       Symbol                        Industry      Ltp  %Chng
0       TITAN               Consumer Durables   3424.0   2.02
1  INDUSINDBK              Financial Services    817.4   1.66
2     SBILIFE              Financial Services   1860.0   1.56
3      MARUTI  Automobile and Auto Components  12543.0   1.44
4   COALINDIA      Oil Gas & Consumable Fuels    379.8   1.36

bottom 5 by % change:
        Symbol                    Industry     Ltp  %Chng
49  ADANIPORTS                    Services  1361.9  -1.94
48    RELIANCE  Oil Gas & Consumable Fuels  1390.0  -1.52
47        INFY      Information Technology  1460.0  -1.38
46    ADANIENT             Metals & Mining  2333.0  -1.29
44   ICICIBANK          Financial Services  1445.0  -1.24


In [ ]:
# Correlation between numeric columns.
corr = df[["Open", "High", "Low", "Ltp", "%Chng", "30 D   %Chng", "365 D %Chng", "Volume"]].corr()
print(corr.round(2))
print("\nWhat to read here: Open/High/Low/Ltp are ~1.0 with each other because they are")
print("four prices of the same stock on the same day - that is not a finding.")
print("The interesting cells are the %Chng columns against Volume and against each other.")

              Open  High   Low   Ltp  %Chng  30 D   %Chng  365 D %Chng  Volume
Open          1.00  1.00  1.00  1.00   0.30          0.11         0.23   -0.49
High          1.00  1.00  1.00  1.00   0.30          0.11         0.23   -0.49
Low           1.00  1.00  1.00  1.00   0.30          0.11         0.23   -0.49
Ltp           1.00  1.00  1.00  1.00   0.31          0.11         0.23   -0.49
%Chng         0.30  0.30  0.30  0.31   1.00          0.04         0.07   -0.19
30 D   %Chng  0.11  0.11  0.11  0.11   0.04          1.00         0.31    0.07
365 D %Chng   0.23  0.23  0.23  0.23   0.07          0.31         1.00   -0.07
Volume       -0.49 -0.49 -0.49 -0.49  -0.19          0.07        -0.07    1.00

What to read here: Open/High/Low/Ltp are ~1.0 with each other because they are
four prices of the same stock on the same day - that is a tautology, not a finding.
The interesting cells are the %Chng columns against Volume and against each other.


In [ ]:
# Sorting: multiple keys, mixed directions, and where NaNs go.
print("sort by Industry (asc) then %Chng (desc):")
print(df.sort_values(["Industry", "%Chng"], ascending=[True, False])
        [["Industry", "Symbol", "%Chng"]].head(8))

print("\nsort_index() sorts by the INDEX, not by a column:")
print(df.set_index("Symbol").sort_index().head(4)[["Ltp", "%Chng"]])

print("\nna_position controls where missing values land:")
demo = pd.Series([3.0, np.nan, 1.0, 2.0])
print("default (last):", demo.sort_values().tolist())
print("na_position='first':", demo.sort_values(na_position="first").tolist())

sort by Industry (asc) then %Chng (desc):
                          Industry      Symbol  %Chng
3   Automobile and Auto Components      MARUTI   1.44
6   Automobile and Auto Components   EICHERMOT   1.12
17  Automobile and Auto Components  BAJAJ-AUTO   0.41
18  Automobile and Auto Components         M&M   0.40
21  Automobile and Auto Components  TATAMOTORS   0.21
23  Automobile and Auto Components  HEROMOTOCO   0.09
41                   Capital Goods         BEL  -0.76
13                    Construction          LT   0.52

sort_index() sorts by the INDEX, not by a column:
               Ltp  %Chng
Symbol                   
ADANIENT    2333.0  -1.29
ADANIPORTS  1361.9  -1.94
APOLLOHOSP  7260.0  -0.66
ASIANPAINT  2434.9  -0.61

na_position controls where missing values land:
default (last): [1.0, 2.0, 3.0, nan]
na_position='first': [nan, 1.0, 2.0, 3.0]


In [15]:
# rank() — turn values into positions. Note how ties are handled.
r = df[["Symbol", "%Chng"]].copy()
#r.loc[5,"%Chng"]=1.36
print(r.sort_values("%Chng", ascending=False).head(6))

r["rank_desc"] = r["%Chng"].rank(ascending=False)                 # 1 = biggest gainer
r["rank_pct"] = r["%Chng"].rank(pct=True).round(3)                # percentile 0..1
r["rank_dense"] = r["%Chng"].rank(method="dense", ascending=False)
print(r.sort_values("rank_desc").head(6))
print("\nmethod= controls ties: 'average' (default), 'min', 'max', 'first', 'dense'.")

       Symbol  %Chng
0       TITAN   2.02
1  INDUSINDBK   1.66
2     SBILIFE   1.56
3      MARUTI   1.44
4   COALINDIA   1.36
5       TRENT   1.35
       Symbol  %Chng  rank_desc  rank_pct  rank_dense
0       TITAN   2.02        1.0      1.00         1.0
1  INDUSINDBK   1.66        2.0      0.98         2.0
2     SBILIFE   1.56        3.0      0.96         3.0
3      MARUTI   1.44        4.0      0.94         4.0
4   COALINDIA   1.36        5.0      0.92         5.0
5       TRENT   1.35        6.0      0.90         6.0

method= controls ties: 'average' (default), 'min', 'max', 'first', 'dense'.


In [ ]:
# Cumulative functions run down the column in order.
top = df.nlargest(6, "%Chng")[["Symbol", "%Chng"]].reset_index(drop=True)
top["cumsum"] = top["%Chng"].cumsum()
top["cummax"] = top["%Chng"].cummax()
print(top)

       Symbol  %Chng  cumsum  cummax
0       TITAN   2.02    2.02    2.02
1  INDUSINDBK   1.66    3.68    2.02
2     SBILIFE   1.56    5.24    2.02
3      MARUTI   1.44    6.68    2.02
4   COALINDIA   1.36    8.04    2.02
5       TRENT   1.35    9.39    2.02


## 4 · Missing data

We add missing data randomly to ur data

In [20]:
rng = np.random.default_rng(42)

dirty = df.copy()
for col, n_missing in [("Volume", 8), ("52W H", 5), ("30 D   %Chng", 6), ("Industry", 3)]:
    idx = rng.choice(dirty.index, size=n_missing, replace=False)
    dirty.loc[idx, col] = np.nan

print("missing values per column (only columns that have any):")
counts = dirty.isna().sum()
print(counts[counts > 0])
print("\nas a percentage of rows:")
print((100 * counts[counts > 0] / len(dirty)).round(1))
print(dirty.isna().mean()*100)
print(dirty.info())

missing values per column (only columns that have any):
Indicative Close    50
Volume               8
52W H                5
30 D   %Chng         6
365 D %Chng          1
Industry             3
dtype: int64

as a percentage of rows:
Indicative Close    100.0
Volume               16.0
52W H                10.0
30 D   %Chng         12.0
365 D %Chng           2.0
Industry              6.0
dtype: float64
Symbol                0.0
Open                  0.0
High                  0.0
Low                   0.0
Prev. Close           0.0
Ltp                   0.0
Indicative Close    100.0
Chng                  0.0
%Chng                 0.0
Volume               16.0
Value                 0.0
52W H                10.0
52W L                 0.0
30 D   %Chng         12.0
365 D %Chng           2.0
Company Name          0.0
Industry              6.0
dtype: float64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 17 columns):
 #   Column            Non-Null Coun

In [ ]:
# isna / notna give you a boolean mask you can count, filter or plot.
print("total missing cells :", dirty.isna().sum().sum())
print("rows with ANY missing value :", dirty.isna().any(axis=1).sum())
print("rows that are COMPLETE      :", dirty.notna().all(axis=1).sum())
print("\nthe rows missing a Volume:")
print(dirty[dirty["Volume"].isna()][["Symbol", "Ltp", "Volume"]])

total missing cells : 73
rows with ANY missing value : 50
rows that are COMPLETE      : 0

the rows missing a Volume:
        Symbol       Ltp  Volume
3       MARUTI  12543.00     NaN
4    COALINDIA    379.80     NaN
20    AXISBANK   1071.10     NaN
29    HDFCLIFE    737.65     NaN
34   NESTLEIND   2264.20     NaN
41         BEL    386.60     NaN
46    ADANIENT   2333.00     NaN
49  ADANIPORTS   1361.90     NaN


### How `NaN` behaves in arithmetic —

- **Aggregations skip it.** `mean`, `sum`, `std` all use `skipna=True` by default, so they
  quietly compute over fewer values than you think.
- **Element-wise arithmetic propagates it.** `NaN + 1` is `NaN`.
- **`count()` counts non-missing values**, which is exactly how you find out how many
  values an aggregation actually used.

In [22]:
v = dirty["Volume"]
print("len(v)            :", len(v))
print("v.count()         :", v.count(), "  <- non-missing only")
print("v.mean()          :", round(v.mean(), 2), "  <- computed over", v.count(), "values")
print("v.mean(skipna=False):", v.mean(skipna=False))
print("v.sum()           :", v.sum(), "  <- treats missing as 0")
print("v.sum()           :", v.sum()/50, "  <- mean over 50 values")
print("\nSo an average over a column with 8 missing values is an average of 42 numbers,")
print("not 50 -  Always check .count() next to .mean().")

len(v)            : 50
v.count()         : 42   <- non-missing only
v.mean()          : 5234187.36   <- computed over 42 values
v.mean(skipna=False): nan
v.sum()           : 219835869.0   <- treats missing as 0
v.sum()           : 4396717.38   <- mean over 50 values

So an average over a column with 8 missing values is an average of 42 numbers,
not 50 -  Always check .count() next to .mean().


In [27]:
print(dirty.head())

       Symbol     Open     High       Low  Prev. Close      Ltp  Indicative Close    Chng  %Chng      Volume    Value     52W H     52W L  \
0       TITAN   3356.3   3443.0   3348.00      3356.30   3424.0               NaN   67.70   2.02    837335.0   285.13   3867.00   2925.00   
1  INDUSINDBK    820.0    848.7    810.05       804.05    817.4               NaN   13.35   1.66  18982770.0  1570.06   1498.00    606.00   
2     SBILIFE   1833.3   1860.4   1818.70      1831.50   1860.0               NaN   28.50   1.56   1251783.0   231.37   1936.00   1372.55   
3      MARUTI  12400.0  12577.0  12370.00     12365.00  12543.0               NaN  178.00   1.44         NaN   292.25  13541.65  10725.00   
4   COALINDIA    376.0    380.7    374.50       374.70    379.8               NaN    5.10   1.36         NaN   258.48    543.55    349.25   

   30 D   %Chng  365 D %Chng                     Company Name                    Industry  
0         -7.29        -1.28               Titan Company Ltd.

In [33]:
# dropna — three different questions.
print("original                     :", dirty.shape)
print("drop rows with ANY missing   :", dirty.dropna().shape)
print("drop rows missing Volume only:", dirty.dropna(subset=["Volume"]).shape)
print("drop rows with <=1 missing   :", dirty.dropna(thresh=15).shape) #Require that many non-NA values
print("drop COLUMNS that are all NaN:", dirty.dropna(axis=1, how="all").shape,
      "  <- this removes 'Indicative Close'")

original                     : (50, 17)
drop rows with ANY missing   : (0, 17)
drop rows missing Volume only: (42, 17)
drop rows with <=1 missing   : (46, 17)
drop COLUMNS that are all NaN: (50, 16)   <- this removes 'Indicative Close'


In [ ]:
# fillna — and the important point that the right filler depends on the column.
filled = dirty.copy()
filled["Volume"] = filled["Volume"].fillna(filled["Volume"].median())   # skewed -> median
filled["30 D   %Chng"] = filled["30 D   %Chng"].fillna(0)               # "no change" is meaningful
filled["Industry"] = filled["Industry"].fillna("Unknown")               # a visible label, not a guess

print("missing after filling:")
c = filled.isna().sum()
print(c[c > 0])
print("\nNote 'Unknown' rather than dropping or guessing the industry:")
print(filled["Industry"].value_counts().tail(3))

missing after filling:
Indicative Close    50
52W H                5
365 D %Chng          1
dtype: int64

Note 'Unknown' rather than dropping or guessing the industry:
Industry
Telecommunication    1
Construction         1
Services             1
Name: count, dtype: int64


In [ ]:
# ffill / bfill carry the last (or next) observation forward. Only sensible on ORDERED data,
# which a table of 50 stocks is not - so here is a small time series to show it properly.
ts = pd.Series([100.0, np.nan, np.nan, 103.0, np.nan, 107.0],
               index=pd.date_range("2025-08-01", periods=6, freq="D"))
print(pd.DataFrame({
    "original": ts,
    "ffill": ts.ffill(),
    "bfill": ts.bfill(),
    "interpolate": ts.interpolate(),
}))
print("\nffill repeats the last known value; interpolate draws a straight line between")
print("known points. For a price series ffill is usually right (a market holiday means")
print("the price did not change); interpolate invents prices that never traded.")
print("Older code writes fillna(method='ffill') - .ffill() is the current spelling.")

            original  ffill  bfill  interpolate
2025-08-01     100.0  100.0  100.0        100.0
2025-08-02       NaN  100.0  103.0        101.0
2025-08-03       NaN  100.0  103.0        102.0
2025-08-04     103.0  103.0  103.0        103.0
2025-08-05       NaN  103.0  107.0        105.0
2025-08-06     107.0  107.0  107.0        107.0

ffill repeats the last known value; interpolate draws a straight line between
known points. For a price series ffill is usually right (a market holiday means
the price did not change); interpolate invents prices that never traded.
Older code writes fillna(method='ffill') - .ffill() is the current spelling.


## 5 · Duplicates

In [ ]:
# Build a frame with deliberate duplicates: the same rows re-appended, plus a same-symbol row.
dupes = pd.concat([df.head(4), df.head(2), df.head(1)], ignore_index=True)
print(dupes[["Symbol", "Ltp", "%Chng"]])

print("\n.duplicated() marks every occurrence AFTER the first:")
print(dupes.duplicated().tolist())
print("keep='last'  marks every occurrence BEFORE the last:", dupes.duplicated(keep="last").tolist())
print("keep=False   marks ALL members of a duplicate set  :", dupes.duplicated(keep=False).tolist())
print("\nhow many duplicate rows:", dupes.duplicated().sum())

       Symbol      Ltp  %Chng
0       TITAN   3424.0   2.02
1  INDUSINDBK    817.4   1.66
2     SBILIFE   1860.0   1.56
3      MARUTI  12543.0   1.44
4       TITAN   3424.0   2.02
5  INDUSINDBK    817.4   1.66
6       TITAN   3424.0   2.02

.duplicated() marks every occurrence AFTER the first:
[False, False, False, False, True, True, True]
keep='last'  marks every occurrence BEFORE the last: [True, True, False, False, True, False, False]
keep=False   marks ALL members of a duplicate set  : [True, True, False, False, True, True, True]

how many duplicate rows: 3


In [ ]:
print("drop_duplicates()                :", dupes.drop_duplicates().shape)
print("drop_duplicates(subset=['Symbol']):", dupes.drop_duplicates(subset=["Symbol"]).shape)
print("...keep='last'                    :",
      dupes.drop_duplicates(subset=["Symbol"], keep="last").shape)
print("\nDeduplicate on the BUSINESS KEY (here, Symbol), not on whole-row equality.")
print("Two rows can legitimately be identical; two rows for the same symbol usually cannot.")

drop_duplicates()                : (4, 17)
drop_duplicates(subset=['Symbol']): (4, 17)
...keep='last'                    : (4, 17)

Deduplicate on the BUSINESS KEY (here, Symbol), not on whole-row equality.
Two rows can legitimately be identical; two rows for the same symbol usually cannot.


In [ ]:
# A useful check on any table that claims to have a key:
print("is Symbol unique in df?      ", df["Symbol"].is_unique)
print("is Symbol unique in dupes?   ", dupes["Symbol"].is_unique)
print("\nduplicated symbols in dupes:")
print(dupes.loc[dupes["Symbol"].duplicated(keep=False), "Symbol"].tolist())

is Symbol unique in df?       True
is Symbol unique in dupes?    False

duplicated symbols in dupes:
['TITAN', 'INDUSINDBK', 'TITAN', 'INDUSINDBK', 'TITAN']


## 6 · String methods — the `.str` accessor

`.str` applies a string operation to every element of a Series at once. It handles `NaN`
for you (missing stays missing) — a plain Python loop would raise.

In [ ]:
name = df["Company Name"]
out = pd.DataFrame({
    "original": name,
    "lower": name.str.lower(),
    "len": name.str.len(),
    "first_word": name.str.split().str[0],
    "has_ltd": name.str.contains("Ltd", case=False, na=False),
})
print(out.head(8).to_string())

                          original                            lower  len first_word  has_ltd
0               Titan Company Ltd.               titan company ltd.   18      Titan     True
1               IndusInd Bank Ltd.               indusind bank ltd.   18   IndusInd     True
2  SBI Life Insurance Company Ltd.  sbi life insurance company ltd.   31        SBI     True
3         Maruti Suzuki India Ltd.         maruti suzuki india ltd.   24     Maruti     True
4                  Coal India Ltd.                  coal india ltd.   15       Coal     True
5                       Trent Ltd.                       trent ltd.   10      Trent     True
6               Eicher Motors Ltd.               eicher motors ltd.   18     Eicher     True
7               Bharti Airtel Ltd.               bharti airtel ltd.   18     Bharti     True


In [35]:
# Filtering with strings.
print("companies with 'Bank' in the name:")
print(df.loc[df["Company Name"].str.contains("bank", case=False, na=False),
             ["Symbol", "Company Name", "Industry"]])

print("\nsymbols starting with 'A':")
print(df.loc[df["Symbol"].str.startswith("A"), "Symbol"].tolist())

print("\nCommon .str methods: lower upper title strip len contains startswith endswith")
print("replace split get extract findall pad zfill cat slice")

companies with 'Bank' in the name:
        Symbol              Company Name            Industry
1   INDUSINDBK        IndusInd Bank Ltd.  Financial Services
10        SBIN       State Bank of India  Financial Services
19   KOTAKBANK  Kotak Mahindra Bank Ltd.  Financial Services
20    AXISBANK            Axis Bank Ltd.  Financial Services
33    HDFCBANK            HDFC Bank Ltd.  Financial Services
44   ICICIBANK           ICICI Bank Ltd.  Financial Services

symbols starting with 'A':
['AXISBANK', 'ASIANPAINT', 'APOLLOHOSP', 'ADANIENT', 'ADANIPORTS']

Common .str methods: lower upper title strip len contains startswith endswith
replace split get extract findall pad zfill cat slice


## 7 · Binning and recoding


In [ ]:
# pd.cut  — FIXED bin edges you choose. Bins can be very unequal in size.
df["PriceBand"] = pd.cut(df["Ltp"],
                         bins=[0, 500, 2000, 5000, np.inf],
                         labels=["<500", "500-2k", "2k-5k", "5k+"])
print(df["PriceBand"].value_counts().sort_index())

# pd.qcut — EQUAL-SIZED buckets, edges chosen from the data (quantiles).
df["PriceQuartile"] = pd.qcut(df["Ltp"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
print("\n", df["PriceQuartile"].value_counts().sort_index())
print("\ncut  answers 'which band is this price in?'   (edges fixed, counts vary)")
print("qcut answers 'is this price high or low RELATIVE to the others?' (counts fixed)")

PriceBand
<500      10
500-2k    23
2k-5k     11
5k+        6
Name: count, dtype: int64

 PriceQuartile
Q1    13
Q2    12
Q3    12
Q4    13
Name: count, dtype: int64

cut  answers 'which band is this price in?'   (edges fixed, counts vary)
qcut answers 'is this price high or low RELATIVE to the others?' (counts fixed)


In [36]:
print(df.head(3))

       Symbol    Open    High      Low  Prev. Close     Ltp  Indicative Close   Chng  %Chng    Volume    Value   52W H    52W L  30 D   %Chng  \
0       TITAN  3356.3  3443.0  3348.00      3356.30  3424.0               NaN  67.70   2.02    837335   285.13  3867.0  2925.00         -7.29   
1  INDUSINDBK   820.0   848.7   810.05       804.05   817.4               NaN  13.35   1.66  18982770  1570.06  1498.0   606.00         -4.33   
2     SBILIFE  1833.3  1860.4  1818.70      1831.50  1860.0               NaN  28.50   1.56   1251783   231.37  1936.0  1372.55          3.18   

   365 D %Chng                     Company Name            Industry  
0        -1.28               Titan Company Ltd.   Consumer Durables  
1       -41.67               IndusInd Bank Ltd.  Financial Services  
2         6.42  SBI Life Insurance Company Ltd.  Financial Services  


In [ ]:
# apply on a Series, apply on a DataFrame, and DataFrame.map for element-wise work.
print("Series.apply — one value in, one value out:")
print(df["Ltp"].apply(lambda p: round(p / 1000, 2)).head(3).tolist())

print("\nDataFrame.apply(axis=1) — one ROW in, one value out:")
spread = df.apply(lambda row: row["High"] - row["Low"], axis=1)
print(spread.head(3).round(2).tolist())

print("\nDataFrame.map — element-wise over EVERY cell (was called .applymap before pandas 2.1):")
print(df[["Open", "High"]].head(3).map(lambda v: f"{v:,.0f}"))

print("\nSpeed note: prefer vectorised expressions to .apply where one exists.")
print("df['High'] - df['Low'] does the same as the axis=1 apply above and is far faster.")

Series.apply — one value in, one value out:
[3.42, 0.82, 1.86]

DataFrame.apply(axis=1) — one ROW in, one value out:
[95.0, 38.65, 41.7]

DataFrame.map — element-wise over EVERY cell (was called .applymap before pandas 2.1):
    Open   High
0  3,356  3,443
1    820    849
2  1,833  1,860

Speed note: prefer vectorised expressions to .apply where one exists.
df['High'] - df['Low'] does the same as the axis=1 apply above and is far faster.


## 8 · Dates and times



In [38]:
symbols = ["INFY", "TCS", "HDFCBANK", "RELIANCE", "ITC", "TITAN", "MARUTI", "SBIN"]
anchor = df.set_index("Symbol").loc[symbols, "Ltp"]

dates = pd.bdate_range(end="2025-08-01", periods=250)      # business days only
rng = np.random.default_rng(7)

frames = []
for sym in symbols:
    shocks = rng.normal(loc=0.0004, scale=0.013, size=len(dates))
    path = anchor[sym] * np.exp(np.cumsum(shocks) - shocks.sum())   # ends at the real Ltp
    frames.append(pd.DataFrame({"Date": dates, "Symbol": sym, "Close": path.round(2)}))

prices = pd.concat(frames, ignore_index=True)
prices["Volume"] = rng.integers(1e5, 5e6, size=len(prices))
print(prices.shape)
prices.head()

(2000, 4)


,Date,Symbol,Close,Volume
0,2024-08-19,INFY,2205.96,4750074
1,2024-08-20,INFY,2215.43,2071287
2,2024-08-21,INFY,2208.43,3243918
3,2024-08-22,INFY,2183.89,3970015
4,2024-08-23,INFY,2171.88,3131147


In [39]:
# Right now 'Date' is a real datetime because bdate_range produced one. Very often it is not:
as_text = prices.head(3).copy()
as_text["Date"] = as_text["Date"].astype(str)
print("dtype when read from a CSV without help:", as_text["Date"].dtype)
print(pd.to_datetime(as_text["Date"]))

print("\nAmbiguous formats need telling. 03/04/2025 is 3 April in India, 4 March in the US:")
print("  dayfirst=True  ->", pd.to_datetime("03/04/2025", dayfirst=True).date())
print("  dayfirst=False ->", pd.to_datetime("03/04/2025", dayfirst=False).date())
print("  explicit format->", pd.to_datetime("03/04/2025", format="%d/%m/%Y").date())
print("\nAlways pass format= if you know it. errors='coerce' turns bad dates into NaT.")
print("  bad value      ->", pd.to_datetime("not a date", errors="coerce"))

dtype when read from a CSV without help: object
0   2024-08-19
1   2024-08-20
2   2024-08-21
Name: Date, dtype: datetime64[ns]

Ambiguous formats need telling. 03/04/2025 is 3 April in India, 4 March in the US:
  dayfirst=True  -> 2025-04-03
  dayfirst=False -> 2025-03-04
  explicit format-> 2025-04-03

Always pass format= if you know it. errors='coerce' turns bad dates into NaT.
  bad value      -> NaT


In [40]:
# The .dt accessor: date parts, as columns.
d = prices.head(5).copy()
d["year"] = d["Date"].dt.year
d["month"] = d["Date"].dt.month
d["month_name"] = d["Date"].dt.month_name()
d["day_name"] = d["Date"].dt.day_name()
d["quarter"] = d["Date"].dt.quarter
d["is_month_end"] = d["Date"].dt.is_month_end
print(d.to_string(index=False))

      Date Symbol   Close  Volume  year  month month_name  day_name  quarter  is_month_end
2024-08-19   INFY 2205.96 4750074  2024      8     August    Monday        3         False
2024-08-20   INFY 2215.43 2071287  2024      8     August   Tuesday        3         False
2024-08-21   INFY 2208.43 3243918  2024      8     August Wednesday        3         False
2024-08-22   INFY 2183.89 3970015  2024      8     August  Thursday        3         False
2024-08-23   INFY 2171.88 3131147  2024      8     August    Friday        3         False


In [41]:
# A DatetimeIndex unlocks partial-string selection - one of pandas' nicest features.
infy = prices[prices["Symbol"] == "INFY"].set_index("Date")["Close"]
print("whole of July 2025:")
print(infy["2025-07"].head(4))
print("...", len(infy["2025-07"]), "trading days in July")
print("\na date range:")
print(infy["2025-07-25":"2025-07-31"])
print("\nlatest value:", infy.iloc[-1], "on", infy.index[-1].date())

whole of July 2025:
Date
2025-07-01    1568.18
2025-07-02    1555.44
2025-07-03    1535.98
2025-07-04    1518.99
Name: Close, dtype: float64
... 23 trading days in July

a date range:
Date
2025-07-25    1529.44
2025-07-28    1553.19
2025-07-29    1553.38
2025-07-30    1510.18
2025-07-31    1497.25
Name: Close, dtype: float64

latest value: 1460.0 on 2025-08-01


In [42]:
print(infy)

Date
2024-08-19    2205.96
2024-08-20    2215.43
2024-08-21    2208.43
2024-08-22    2183.89
2024-08-23    2171.88
               ...   
2025-07-28    1553.19
2025-07-29    1553.38
2025-07-30    1510.18
2025-07-31    1497.25
2025-08-01    1460.00
Name: Close, Length: 250, dtype: float64


In [ ]:
# resample — groupby for time. 'ME' = month end, 'W' = week, 'QE' = quarter end.
monthly = infy.resample("ME").agg(["first", "max", "min", "last", "count"])
monthly.columns = ["Open", "High", "Low", "Close", "TradingDays"]
print("monthly OHLC from daily closes:")
print(monthly.round(2))

print("\nweekly mean, last 5 weeks:")
print(infy.resample("W").mean().round(2).tail())

monthly OHLC from daily closes:
               Open     High      Low    Close  TradingDays
Date                                                       
2024-08-31  2205.96  2215.43  2144.92  2156.41           10
2024-09-30  2171.05  2185.89  1862.22  1862.22           21
2024-10-31  1851.43  1879.17  1801.34  1868.07           23
2024-11-30  1864.24  1934.48  1829.55  1860.44           21
2024-12-31  1877.94  1944.53  1846.00  1912.93           22
2025-01-31  1922.34  1922.34  1805.94  1841.02           23
2025-02-28  1843.90  1843.90  1687.47  1718.47           20
2025-03-31  1743.89  1743.89  1631.60  1632.05           21
2025-04-30  1639.87  1670.82  1541.90  1561.72           22
2025-05-31  1556.23  1762.70  1553.79  1762.70           22
2025-06-30  1797.77  1797.77  1568.49  1587.84           21
2025-07-31  1568.18  1568.18  1497.25  1497.25           23
2025-08-31  1460.00  1460.00  1460.00  1460.00            1

weekly mean, last 5 weeks:
Date
2025-07-06    1553.29
2025-07-13   

In [ ]:
# shift, pct_change and rolling - the three workhorses of time-series analysis.
w = pd.DataFrame({"Close": infy})
w["prev_close"] = w["Close"].shift(1)                 # yesterday's close on today's row
w["daily_ret"] = w["Close"].pct_change() * 100        # same as (Close/prev - 1) * 100
w["ma20"] = w["Close"].rolling(20).mean()             # 20-day moving average
w["vol20"] = w["Close"].pct_change().rolling(20).std() * 100
w["cum_ret"] = ((1 + w["Close"].pct_change()).cumprod() - 1) * 100

print(w.tail(6).round(3).to_string())
print("\nNote the NaNs at the START of the column - rolling(20) needs 20 observations")
print("before it can produce the first value. shift(1) loses the first row for the same reason.")
print("first non-null ma20 at row:", w["ma20"].notna().idxmax().date())

              Close  prev_close  daily_ret      ma20  vol20  cum_ret
Date                                                                
2025-07-25  1529.44     1534.62     -0.338  1539.026  1.101  -30.668
2025-07-28  1553.19     1529.44      1.553  1537.294  1.156  -29.591
2025-07-29  1553.38     1553.19      0.012  1536.554  1.124  -29.583
2025-07-30  1510.18     1553.38     -2.781  1534.290  1.272  -31.541
2025-07-31  1497.25     1510.18     -0.856  1532.354  1.257  -32.127
2025-08-01  1460.00     1497.25     -2.488  1529.404  1.349  -33.816

Note the NaNs at the START of the column - rolling(20) needs 20 observations
before it can produce the first value. shift(1) loses the first row for the same reason.
first non-null ma20 at row: 2024-09-13


## 9 · Reshaping — long vs wide

`prices` is in **long** format: one row per (Date, Symbol). Most analysis wants **wide**:
one row per Date, one column per Symbol. Converting between the two is what this section is.

In [ ]:
# pivot: long -> wide. Requires the index/column pair to be UNIQUE (no aggregation).
wide = prices.pivot(index="Date", columns="Symbol", values="Close")
print("long:", prices.shape, " -> wide:", wide.shape)
print(wide.tail(4).round(2).to_string())

long: (2000, 4)  -> wide: (250, 8)
Symbol      HDFCBANK     INFY     ITC    MARUTI  RELIANCE    SBIN      TCS    TITAN
Date                                                                               
2025-07-29   1941.12  1553.38  419.01  12438.00   1402.18  820.72  3100.89  3413.35
2025-07-30   1959.80  1510.18  425.00  12166.89   1395.46  822.25  3059.05  3447.89
2025-07-31   1943.41  1497.25  427.06  12358.62   1404.43  809.25  3101.94  3431.43
2025-08-01   1981.00  1460.00  413.80  12543.00   1390.00  800.50  3060.00  3424.00


In [ ]:
# Now that it is wide, cross-sectional work becomes easy.
print("correlation of daily returns between symbols:")
print(wide.pct_change().corr().round(2).to_string())

correlation of daily returns between symbols:
Symbol    HDFCBANK  INFY   ITC  MARUTI  RELIANCE  SBIN   TCS  TITAN
Symbol                                                             
HDFCBANK      1.00  0.04  0.05    0.06      0.01  0.05 -0.12  -0.04
INFY          0.04  1.00 -0.06   -0.06     -0.02 -0.11  0.12   0.08
ITC           0.05 -0.06  1.00   -0.00     -0.01  0.09  0.05  -0.05
MARUTI        0.06 -0.06 -0.00    1.00      0.08  0.08  0.04   0.08
RELIANCE      0.01 -0.02 -0.01    0.08      1.00  0.05 -0.03  -0.13
SBIN          0.05 -0.11  0.09    0.08      0.05  1.00 -0.13  -0.23
TCS          -0.12  0.12  0.05    0.04     -0.03 -0.13  1.00  -0.01
TITAN        -0.04  0.08 -0.05    0.08     -0.13 -0.23 -0.01   1.00


In [ ]:
# melt: wide -> long. The inverse of pivot.
back = wide.reset_index().melt(id_vars="Date", var_name="Symbol", value_name="Close")
print("melted back to long:", back.shape)
print(back.head(3))
print("\nround-trips exactly?",
      back.sort_values(["Symbol", "Date"]).reset_index(drop=True)["Close"]
          .equals(prices.sort_values(["Symbol", "Date"]).reset_index(drop=True)["Close"]))

melted back to long: (2000, 3)
        Date    Symbol    Close
0 2024-08-19  HDFCBANK  2080.25
1 2024-08-20  HDFCBANK  2040.73
2 2024-08-21  HDFCBANK  2015.77

round-trips exactly? True


In [ ]:
# pivot_table: like pivot, but it AGGREGATES, so duplicates are fine.
prices["Month"] = prices["Date"].dt.to_period("M").astype(str)
pt = prices.pivot_table(index="Month", columns="Symbol", values="Close", aggfunc="mean")
print("mean monthly close:")
print(pt.tail(4).round(1).to_string())

print("\nseveral aggregations at once, with row/column totals:")
pt2 = prices.pivot_table(index="Symbol", values=["Close", "Volume"],
                         aggfunc={"Close": "mean", "Volume": ["min", "max"]})
print(pt2.round(0).to_string())

mean monthly close:
Symbol   HDFCBANK    INFY    ITC   MARUTI  RELIANCE   SBIN     TCS   TITAN
Month                                                                     
2025-05    1870.9  1629.4  384.7  12723.6    1251.6  816.4  3767.5  3424.1
2025-06    2035.8  1674.8  383.8  12674.0    1377.8  797.8  3529.2  3450.6
2025-07    1922.9  1535.1  401.2  12356.8    1380.7  808.7  3219.4  3441.6
2025-08    1981.0  1460.0  413.8  12543.0    1390.0  800.5  3060.0  3424.0

several aggregations at once, with row/column totals:


            Close   Volume        
             mean      max     min
Symbol                            
HDFCBANK   1842.0  4996749  110188
INFY       1774.0  4992147  110622
ITC         413.0  4989014  101266
MARUTI    14374.0  4996873  100888
RELIANCE   1186.0  4985872  148572
SBIN        712.0  4995538  133233
TCS        3826.0  4976036  122887
TITAN      3370.0  4955249  118314


In [ ]:
# pivot_table with margins (grand totals) on the snapshot data.
tab = df.pivot_table(index="Industry", columns="Sign", values="Symbol",
                     aggfunc="count", fill_value=0, margins=True, margins_name="Total")
print(tab.to_string())

Sign                            Advance  Decline  Total
Industry                                               
Automobile and Auto Components        6        0      6
Capital Goods                         0        1      1
Construction                          1        0      1
Construction Materials                2        0      2
Consumer Durables                     1        1      2
Consumer Services                     1        1      2
Fast Moving Consumer Goods            0        4      4
Financial Services                    8        4     12
Healthcare                            0        4      4
Information Technology                2        3      5
Metals & Mining                       0        4      4
Oil Gas & Consumable Fuels            1        2      3
Power                                 1        1      2
Services                              0        1      1
Telecommunication                     1        0      1
Total                                24       26

In [ ]:
# crosstab: a frequency table of two categoricals. Shorthand for the pivot_table above.
ct = pd.crosstab(df["Industry"], df["Sign"])
print(ct.to_string())

print("\nnormalised by row - what share of each industry advanced?")
print(pd.crosstab(df["Industry"], df["Sign"], normalize="index").round(2).to_string())

Sign                            Advance  Decline
Industry                                        
Automobile and Auto Components        6        0
Capital Goods                         0        1
Construction                          1        0
Construction Materials                2        0
Consumer Durables                     1        1
Consumer Services                     1        1
Fast Moving Consumer Goods            0        4
Financial Services                    8        4
Healthcare                            0        4
Information Technology                2        3
Metals & Mining                       0        4
Oil Gas & Consumable Fuels            1        2
Power                                 1        1
Services                              0        1
Telecommunication                     1        0

normalised by row - what share of each industry advanced?
Sign                            Advance  Decline
Industry                                        
Automobile

In [ ]:
# stack / unstack move data between the index and the columns.
by_two = df.groupby(["Industry", "Sign"], observed=True)["Symbol"].count()
print("groupby with TWO keys gives a MultiIndex Series:")
print(by_two.head(8), "\n")

print("unstack() pushes the LAST index level up into the columns:")
print(by_two.unstack(fill_value=0).head().to_string())

print("\nstack() is the inverse - columns back down into the index:")
print(by_two.unstack(fill_value=0).stack().head(6))

groupby with TWO keys gives a MultiIndex Series:
Industry                        Sign   
Automobile and Auto Components  Advance    6
Capital Goods                   Decline    1
Construction                    Advance    1
Construction Materials          Advance    2
Consumer Durables               Advance    1
                                Decline    1
Consumer Services               Advance    1
                                Decline    1
Name: Symbol, dtype: int64 

unstack() pushes the LAST index level up into the columns:
Sign                            Advance  Decline
Industry                                        
Automobile and Auto Components        6        0
Capital Goods                         0        1
Construction                          1        0
Construction Materials                2        0
Consumer Durables                     1        1

stack() is the inverse - columns back down into the index:
Industry                        Sign   
Automobile and Auto 

## 10 · `groupby` in depth

`pandas1` used `groupby` with one key and one aggregation. Four things it did not cover:
**multiple keys**, **named aggregation**, **`transform`**, and **`filter`**.

In [47]:
# The mental model: SPLIT the frame into groups, APPLY a function to each, COMBINE the results.
g = df.groupby("Industry", observed=True)
print(g.get_group("Information Technology").head(3))
#print(pd.DataFrame(g))
print("number of groups:", g.ngroups)
print("\ngroup sizes (size counts ROWS; count counts NON-NULL values per column):")
print(pd.DataFrame({"size": g.size(), "count_of_Ltp": g["Ltp"].count()}).head())

print("\nyou can look at one group directly:")
print(g.get_group("Information Technology")[["Symbol", "Ltp", "%Chng"]])

     Symbol     Open     High      Low  Prev. Close      Ltp  Indicative Close  Chng  %Chng   Volume   Value   52W H    52W L  30 D   %Chng  \
12    TECHM  1470.10  1487.80  1461.20      1475.00  1483.00               NaN   8.0   0.54  2929677  433.47  1807.7  1209.40        -10.26   
15  HCLTECH  1479.90  1485.40  1471.20      1474.40  1480.90               NaN   6.5   0.44  2614394  387.00  2012.2  1302.75        -14.09   
27    WIPRO   246.05   246.95   244.53       246.05   245.45               NaN  -0.6  -0.24  7383968  181.33   324.6   228.00         -8.94   

    365 D %Chng           Company Name                Industry  
12        -1.48     Tech Mahindra Ltd.  Information Technology  
15        -7.87  HCL Technologies Ltd.  Information Technology  
27       -51.03             Wipro Ltd.  Information Technology  
number of groups: 15

group sizes (size counts ROWS; count counts NON-NULL values per column):
                                size  count_of_Ltp
Industry             

In [ ]:
# Named aggregation - the clearest syntax, and you control the output column names.
summary = df.groupby("Industry", observed=True).agg(
    n_companies=("Symbol", "count"),
    avg_change=("%Chng", "mean"),
    median_change=("%Chng", "median"),
    best=("%Chng", "max"),
    total_value=("Value", "sum"),
).round(2).sort_values("avg_change", ascending=False)
print(summary.to_string())

                                n_companies  avg_change  median_change  best  total_value
Industry                                                                                 
Telecommunication                         1        0.82           0.82  0.82       996.80
Consumer Durables                         2        0.70           0.70  2.02       445.48
Automobile and Auto Components            6        0.61           0.40  1.44      2092.47
Construction                              1        0.52           0.52  0.52       421.41
Construction Materials                    2        0.38           0.38  0.57       548.48
Financial Services                       12        0.32           0.39  1.66      9496.31
Consumer Services                         2        0.14           0.14  1.35       967.35
Power                                     2       -0.10          -0.10  0.50       773.66
Oil Gas & Consumable Fuels                3       -0.13          -0.23  1.36      1781.65
Informatio

In [ ]:
# Multiple grouping keys, and as_index=False to get a flat frame straight away.
two_key = df.groupby(["Industry", "Sign"], observed=True, as_index=False).agg(
    n=("Symbol", "count"), avg=("%Chng", "mean")).round(2)
print(two_key.head(8).to_string(index=False))

# A custom function is fine too.
def spread(s):
    return s.max() - s.min()

print("\ncustom aggregation (range of %Chng within each industry):")
print(df.groupby("Industry", observed=True)["%Chng"].agg(spread).round(2).sort_values().tail())

                      Industry    Sign  n   avg
Automobile and Auto Components Advance  6  0.61
                 Capital Goods Decline  1 -0.76
                  Construction Advance  1  0.52
        Construction Materials Advance  2  0.38
             Consumer Durables Advance  1  2.02
             Consumer Durables Decline  1 -0.61
             Consumer Services Advance  1  1.35
             Consumer Services Decline  1 -1.08

custom aggregation (range of %Chng within each industry):
Industry
Information Technology        1.92
Consumer Services             2.43
Consumer Durables             2.63
Oil Gas & Consumable Fuels    2.88
Financial Services            2.90
Name: %Chng, dtype: float64


### `agg` vs `transform` vs `filter` — the distinction that matters

| | Returns | Shape | Use it for |
|---|---|---|---|
| `.agg()` | one value per group | **n_groups** rows | summaries |
| `.transform()` | one value per **row**, broadcast from its group | **n_rows** rows | new columns |
| `.filter()` | whole groups, kept or dropped | subset of rows | removing small/odd groups |

In [ ]:
# transform broadcasts the group's statistic back onto every row of that group.
t = df[["Symbol", "Industry", "%Chng"]].copy()
t["industry_mean"] = df.groupby("Industry", observed=True)["%Chng"].transform("mean").round(2)
t["vs_industry"] = (t["%Chng"] - t["industry_mean"]).round(2)
t["industry_zscore"] = df.groupby("Industry", observed=True)["%Chng"].transform(
    lambda s: (s - s.mean()) / s.std()).round(2)
print(t.sort_values("vs_industry", ascending=False).head(8).to_string(index=False))
print("\nagg would have given 15 rows. transform gives 50 - one per original row -")
print("which is what you need to add a column. This is the single most useful groupby trick.")

    Symbol                       Industry  %Chng  industry_mean  vs_industry  industry_zscore
 COALINDIA     Oil Gas & Consumable Fuels   1.36          -0.13         1.49             1.03
INDUSINDBK             Financial Services   1.66           0.32         1.34             1.59
     TITAN              Consumer Durables   2.02           0.70         1.32             0.71
   SBILIFE             Financial Services   1.56           0.32         1.24             1.47
     TRENT              Consumer Services   1.35           0.14         1.21             0.71
    MARUTI Automobile and Auto Components   1.44           0.61         0.83             1.53
     TECHM         Information Technology   0.54          -0.22         0.76             0.98
   HCLTECH         Information Technology   0.44          -0.22         0.66             0.85

agg would have given 15 rows. transform gives 50 - one per original row -
which is what you need to add a column. This is the single most useful groupby 

In [ ]:
# filter keeps or drops WHOLE groups based on a test of the group.
big = df.groupby("Industry", observed=True).filter(lambda gr: len(gr) >= 4)
print("industries with at least 4 companies:")
print(big["Industry"].value_counts())
print("\nrows kept:", len(big), "of", len(df))

industries with at least 4 companies:
Industry
Financial Services                12
Automobile and Auto Components     6
Information Technology             5
Metals & Mining                    4
Fast Moving Consumer Goods         4
Healthcare                         4
Name: count, dtype: int64

rows kept: 35 of 50


In [ ]:
# groupby also works on the time-series frame, and combines with resample.
prices["Year"] = prices["Date"].dt.year
print("mean close per symbol per year:")
print(prices.groupby(["Symbol", "Year"])["Close"].mean().unstack().round(1).to_string())

print("\nmonthly mean close for every symbol, via groupby + resample:")
print(prices.set_index("Date").groupby("Symbol")["Close"].resample("ME").mean()
            .unstack(0).tail(3).round(1).to_string())

mean close per symbol per year:


Year         2024     2025
Symbol                    
HDFCBANK   1824.0   1852.6
INFY       1936.3   1671.1
ITC         428.8    402.7
MARUTI    15255.7  13814.7
RELIANCE   1143.5   1213.2
SBIN        628.0    764.9
TCS        3858.3   3805.7
TITAN      3307.8   3410.1

monthly mean close for every symbol, via groupby + resample:


Symbol      HDFCBANK    INFY    ITC   MARUTI  RELIANCE   SBIN     TCS   TITAN
Date                                                                         
2025-06-30    2035.8  1674.8  383.8  12674.0    1377.8  797.8  3529.2  3450.6
2025-07-31    1922.9  1535.1  401.2  12356.8    1380.7  808.7  3219.4  3441.6
2025-08-31    1981.0  1460.0  413.8  12543.0    1390.0  800.5  3060.0  3424.0


## 11 · Reading and writing files

`read_csv` has around fifty parameters. These six cover almost everything you will need.

In [ ]:
# to_csv, then read it back with a realistic set of options.
tmp = Path("nifty_clean_demo.csv")
df[["Symbol", "Company Name", "Industry", "Ltp", "%Chng", "Volume"]].to_csv(tmp, index=False)
print("wrote", tmp, f"({tmp.stat().st_size:,} bytes)")
print(tmp.read_text().split("\n")[0])
print(tmp.read_text().split("\n")[1])

wrote nifty_clean_demo.csv (3,565 bytes)


Symbol,Company Name,Industry,Ltp,%Chng,Volume
TITAN,Titan Company Ltd.,Consumer Durables,3424.0,2.02,837335


In [ ]:
back = pd.read_csv(
    tmp,
    index_col="Symbol",                 # use this column as the index
    usecols=["Symbol", "Industry", "Ltp", "%Chng", "Volume"],   # read only what you need
    dtype={"Industry": "category"},     # set dtypes at read time, not after
    na_values=["-", "NA", "n/a", ""],   # extra strings that mean "missing"
)
print(back.dtypes, "\n")
print(back.head(3))
print("\nnrows=5 reads just the first 5 data rows - use it to peek at a huge file:")
print(pd.read_csv(tmp, nrows=5)[["Symbol", "Ltp"]].to_string(index=False))

Industry    category
Ltp          float64
%Chng        float64
Volume         int64
dtype: object 

                      Industry     Ltp  %Chng    Volume
Symbol                                                 
TITAN        Consumer Durables  3424.0   2.02    837335
INDUSINDBK  Financial Services   817.4   1.66  18982770
SBILIFE     Financial Services  1860.0   1.56   1251783

nrows=5 reads just the first 5 data rows - use it to peek at a huge file:
    Symbol     Ltp
     TITAN  3424.0
INDUSINDBK   817.4
   SBILIFE  1860.0
    MARUTI 12543.0
 COALINDIA   379.8


In [ ]:
# parse_dates does the datetime conversion during the read.
ptmp = Path("prices_demo.csv")
prices[["Date", "Symbol", "Close"]].to_csv(ptmp, index=False)

no_parse = pd.read_csv(ptmp)
with_parse = pd.read_csv(ptmp, parse_dates=["Date"])
print("without parse_dates, Date is:", no_parse["Date"].dtype)
print("with    parse_dates, Date is:", with_parse["Date"].dtype)

print("\nOther formats, same idea:")
print("  pd.read_excel(path, sheet_name=0)     /  df.to_excel(path, index=False)")
print("  pd.read_json(path)                    /  df.to_json(path, orient='records')")
print("  pd.read_parquet(path)                 /  df.to_parquet(path)   <- best for large data")

# tidy up the files we just wrote
tmp.unlink(); ptmp.unlink()
print("\ndemo files removed")

without parse_dates, Date is: object
with    parse_dates, Date is: datetime64[ns]

Other formats, same idea:
  pd.read_excel(path, sheet_name=0)     /  df.to_excel(path, index=False)
  pd.read_json(path)                    /  df.to_json(path, orient='records')
  pd.read_parquet(path)                 /  df.to_parquet(path)   <- best for large data

demo files removed


## 12 · `query()`, categoricals, and memory

In [ ]:
# query() - boolean selection written as a string. Often much easier to read.
print("boolean mask style:")
print(df[(df["%Chng"] > 1) & (df["Ltp"] < 2000)][["Symbol", "Ltp", "%Chng"]].head(4))

print("\nquery() style - same result:")
print(df.query("`%Chng` > 1 and Ltp < 2000")[["Symbol", "Ltp", "%Chng"]].head(4))

threshold = 1.5
print("\n@ refers to a Python variable:")
print(df.query("`%Chng` > @threshold")[["Symbol", "%Chng"]].to_string(index=False))
print("\nBacktick a column name that contains spaces or symbols. query() is slower on small")
print("frames and faster on very large ones; on readability it usually wins.")

boolean mask style:
       Symbol     Ltp  %Chng
1  INDUSINDBK   817.4   1.66
2     SBILIFE  1860.0   1.56
4   COALINDIA   379.8   1.36

query() style - same result:
       Symbol     Ltp  %Chng
1  INDUSINDBK   817.4   1.66
2     SBILIFE  1860.0   1.56
4   COALINDIA   379.8   1.36

@ refers to a Python variable:
    Symbol  %Chng
     TITAN   2.02
INDUSINDBK   1.66
   SBILIFE   1.56

Backtick a column name that contains spaces or symbols. query() is slower on small
frames and faster on very large ones; on readability it usually wins.


In [ ]:
# Categorical dtype: for repeated text with few distinct values.
before = df["Industry"].memory_usage(deep=True)
cat = df["Industry"].astype("category")
after = cat.memory_usage(deep=True)

print(f"Industry as object   : {before:,} bytes")
print(f"Industry as category : {after:,} bytes   ({100*(before-after)/before:.0f}% smaller)")
print("\ncategories:", list(cat.cat.categories[:5]), "...")
print("stored internally as small integer codes:", cat.cat.codes.head(5).tolist())

print("\nFull frame memory:")
print(df.memory_usage(deep=True).sort_values(ascending=False).head(6))
print(f"\ntotal: {df.memory_usage(deep=True).sum():,} bytes")

Industry as object   : 3,542 bytes
Industry as category : 1,731 bytes   (51% smaller)

categories: ['Automobile and Auto Components', 'Capital Goods', 'Construction', 'Construction Materials', 'Consumer Durables'] ...
stored internally as small integer codes: [4, 7, 7, 0, 11]

Full frame memory:
Company Name     3482
Industry         3410
Symbol           2819
Sign             2800
PriceBand         436
PriceQuartile     426
dtype: int64

total: 19,105 bytes
